# 14 — Capstone: design a support desk

**Both** — every rule you give an agent either steers it or fences it in. This notebook is about putting
each rule in the right place.

When you finish you can:

- decide where a rule should live, with one table: instructions, output type + validator, filtered
  toolset, hook, approval, usage limits or history;
- turn a plain-English spec into an agent where each requirement sits in exactly one place;
- prove each requirement with one `assert`, offline;
- run the finished desk on your real `MODEL`.

> This notebook runs **keyless**, except one live cell near the end that uses your `MODEL` only if a key
> is present. There is one small new option (`Case(evaluators=...)` in the eval). Every other piece
> comes from 01–13. The new skill is design.

In [ ]:
from learningdantic import setup

MODEL = setup()

## Where does this rule live?

You have met seven places a rule can live. Each one suits some rules better than others:

| A rule that says… | Lives in | Steer or scope | Met in |
|---|---|---|---|
| how to talk, and what to do first ("look the order up before you answer") | `instructions` | Steer | 01, 05 |
| what shape the answer has, and what it must never say | `output_type` + an output validator | Steer | 03 |
| which tools it may see, maybe depending on who is asking | a `.filtered` toolset | Scope | 09 |
| check, log or block a tool call while the run happens | a `Hooks` guardrail | Scope | 10 |
| a person must say yes before it happens | `requires_approval` | Scope | 10 |
| how much one job may spend | `UsageLimits` | Scope | 10, 11 |
| how much of the conversation it remembers | message history + `ProcessHistory` | Scope | 07, 13 |

Two habits help you pick a row:

- **The costlier a broken rule, the more it belongs in code.** Only the first row is a request, and a
  model can ignore a request. Every row below it is checked by your Python code, so the model cannot talk
  its way past it (the validator steers; the other rows scope). "Be friendly" is fine as an instruction.
  "Never refund without a person's OK" is not.
- **Each row has its own proof.** A rule checked in code (rows 2–7) gets an `assert` that runs offline.
  An instruction (row 1) can only be proved offline as far as "the model was told". Whether a real model
  *follows* it is what an eval measures (12).

A `Capability` (10) is not a row of its own. It is a box that carries rows (instructions, toolsets,
hooks), so a second agent can reuse them in one line.

## The spec

Acme sells mugs (you met its support bot in 12). Here is the spec for its support desk, written by
someone who doesn't code. The two right-hand columns are the design work: where each requirement lives,
and how we will prove it.

| # | Requirement | Lives in | Proof (offline) |
|---|---|---|---|
| R1 | Look the order up first, ask billing about money, and end every message with "Need more help? Reply HELP." | `instructions` | the model was shown the rule |
| R2 | Every answer has a category and a message, and never promises a refund. | `output_type=Reply` + a validator | a promise is sent back and fixed |
| R3 | Customers never see the refund tool; staff do. | `.filtered` on `ctx.deps.role` | the customer's tool list |
| R4 | Every tool call goes into the ticket's audit log. | a `Hooks` guardrail | the log after a run |
| R5 | A refund only happens after a person approves it. | `requires_approval=True` | the run pauses |
| R6 | One ticket costs at most 6 model requests, specialists included. | `UsageLimits(request_limit=6)` | a stuck model is stopped |

R1 and R2 both touch refunds, but in different rows. The soft part ("ask billing") is steering. The
hard part ("never promise") is checked in code.

## The parts

First the parts the requirements plug into, all from earlier notebooks:

- `Ticket` is the deps (05): who is writing in, their role, and the audit log for R4.
  `field(default_factory=list)` gives every ticket its own empty list (10).
- `billing_expert` is a specialist (11). The refund policy lives in its instructions and nowhere else.
- `desk_tools` is a toolset (09) with three tools. `ask_billing` runs the specialist, so it is
  `async def`. It forwards `usage` and `usage_limits`, so R6 counts the specialist too (11).
  `issue_refund` carries `requires_approval=True`: that is R5, written on the tool itself.

Every agent here is built on a `TestModel` stand-in, so the notebook runs with no key. The live cell
swaps in your `MODEL` with `override` (08, 12). `TestModel(custom_output_text=...)` makes the stand-in
specialist answer with a fixed sentence (12).

In [ ]:
from dataclasses import dataclass, field

from pydantic_ai import Agent, FunctionToolset, RunContext
from pydantic_ai.models.test import TestModel


@dataclass
class Ticket:
    customer: str
    role: str = "customer"  # "customer" or "staff"
    audit_log: list[str] = field(default_factory=list)


ORDERS = {1042: "1 blue mug, delivered on Monday", 2077: "2 white mugs, in transit"}

billing_expert = Agent(
    TestModel(custom_output_text="A teammate reviews every refund request within 5 days."),
    name="billing_expert",
    instructions=(
        "You are Acme's billing expert. Refund policy: a teammate reviews every refund request "
        "within 5 days, and damaged mugs are replaced or refunded after that review. "
        "Never promise a refund. Answer in one sentence."
    ),
)

desk_tools = FunctionToolset()


@desk_tools.tool_plain
def lookup_order(order_id: int) -> str:
    """Look up one order by its number."""
    return ORDERS.get(order_id, f"There is no order {order_id}.")


@desk_tools.tool
async def ask_billing(ctx: RunContext, question: str) -> str:
    """Ask the billing expert about refunds, charges and invoices."""
    answer = await billing_expert.run(question, usage=ctx.usage, usage_limits=ctx.usage_limits)
    return answer.output


@desk_tools.tool_plain(requires_approval=True)  # R5
def issue_refund(order_id: int) -> str:
    """Refund an order. Staff only."""
    return f"Refunded order {order_id}."

> ✅ **Checkpoint** — no output. If you re-run this cell, re-run the next one too, so the desk
> picks up the new toolset.

## The desk: six requirements, six places

Now put the desk together. Each requirement goes into exactly one place, marked with a `# R…` comment:

- **R1** is the `instructions` string.
- **R2** is `output_type` plus `@desk.output_validator` (03). The validator is a plain `def`.
  `raise ModelRetry(...)` sends your message back to the model so it can fix its answer (03). The list
  also holds `DeferredToolRequests`, which R5 needs to pause a run (10).
- **R3** is `.filtered` reading `ctx.deps.role` (09).
- **R4** is a `Hooks` capability with a `before_tool_execute` hook (10).
- **R6** is a `UsageLimits` passed on every run. `Agent(...)` has no argument for it, so `handle()` is
  the desk's one front door, and it always passes the limits. `handle` awaits `desk.run`, so it is
  `async def` (the one async rule).

In [ ]:
from typing import Literal

from pydantic import BaseModel
from pydantic_ai import DeferredToolRequests, ModelRetry, UsageLimits
from pydantic_ai.capabilities import Hooks

RULES = (  # R1
    "You are the support desk for Acme, a shop that sells mugs. "
    "Look up the order before you answer about it. "
    "For any refund or charge question, ask billing and pass its answer on. "
    "End every message with: Need more help? Reply HELP."
)
PROMISE_WORDS = ["guarantee", "full refund"]


class Reply(BaseModel):  # R2
    category: Literal["billing", "shipping", "other"]
    message: str


audit = Hooks()


@audit.on.before_tool_execute  # R4
def log_call(ctx, call, tool_def, args):
    ctx.deps.audit_log.append(call.tool_name)
    return args


desk = Agent(
    TestModel(),  # a stand-in; the live cell swaps in your MODEL
    name="desk",
    deps_type=Ticket,
    instructions=RULES,  # R1
    output_type=[Reply, DeferredToolRequests],  # R2 (DeferredToolRequests lets R5 pause a run)
    toolsets=[  # R3
        desk_tools.filtered(lambda ctx, td: td.name != "issue_refund" or ctx.deps.role == "staff")
    ],
    capabilities=[audit],  # R4
)


@desk.output_validator  # R2
def no_refund_promise(output: Reply) -> Reply:
    found = [word for word in PROMISE_WORDS if word in output.message.lower()]
    if found:
        raise ModelRetry(
            f"Never promise a refund (you wrote {found}). Say a teammate reviews it within 5 days."
        )
    return output


LIMITS = UsageLimits(request_limit=6)  # R6


async def handle(text: str, ticket: Ticket):
    """The desk's one front door: every ticket runs with the same limits."""
    return await desk.run(text, deps=ticket, usage_limits=LIMITS)

> ✅ **Checkpoint** — no output. Read the cell once more and find `# R1`–`# R4` and `# R6`. R5
> sits on `issue_refund` in the parts cell. When the spec changes, you now know which line to touch.

## R1 — instructions: prove the model was told

A fake model can't prove a rule is *followed*, only that it *arrived*. This scripted desk model (a
`FunctionModel`, 08) keeps a copy of the instructions it was shown (`info.instructions`, as in 10), then
plays a good desk: it looks the order up, asks billing when the message is about money, and replies. We
reuse it for R4 and for the eval below.

`info.output_tools[0].name` is the name of the `Reply` output tool, so the script ends the run with a
`Reply`. `messages[0].parts[-1].content` is the customer's message.

In [ ]:
from pydantic_ai.messages import ModelResponse, ToolCallPart
from pydantic_ai.models.function import FunctionModel

shown = []  # the instructions the fake model saw, one entry per request


def fake_desk(messages, info):
    """A scripted desk model: look up order 1042, ask billing about money, then reply."""
    shown.append(info.instructions)
    text = messages[0].parts[-1].content
    money = "money" in text or "refund" in text
    if len(messages) == 1:  # the first request: call the tools
        calls = [ToolCallPart("lookup_order", {"order_id": 1042})]
        if money:
            calls.append(ToolCallPart("ask_billing", {"question": text}))
        return ModelResponse(parts=calls)
    reply = {
        "category": "billing" if money else "shipping",
        "message": "A teammate will review it within 5 days. Need more help? Reply HELP.",
    }
    return ModelResponse(parts=[ToolCallPart(info.output_tools[0].name, reply)])


ada = Ticket("Ada")
with desk.override(model=FunctionModel(fake_desk)):
    result = await handle("My mug from order 1042 arrived cracked. I want my money back.", ada)
print(result.output)
print("requests:", result.usage.requests)

assert "Reply HELP" in shown[0]  # R1: the rule reached the model

> ✅ **Checkpoint** — `category='billing' message='A teammate will review it within 5 days. Need
> more help? Reply HELP.'` and `requests: 3`: the desk asked its model twice, and the billing expert
> once. The assert passed, so R1 reached the model. Whether a real model obeys it is for the eval and the
> live cell.

## R2 — output type + validator: a promise gets sent back

`Reply` fixes the shape: the run can only end with a category from the list and a message. The
validator adds a rule about the content. This scripted model promises a full refund first. The validator
sends that back with `ModelRetry`, and the model's second try is fine. Retries are a budget: one by
default (03).

In [ ]:
from pydantic_ai.messages import RetryPromptPart


def promise_then_fix(messages, info):
    """A scripted model that promises a refund, then fixes its reply after the retry."""
    retried = any(isinstance(part, RetryPromptPart) for part in messages[-1].parts)
    message = (
        "A teammate will review it within 5 days." if retried else "We guarantee a full refund!"
    )
    reply = {"category": "billing", "message": message + " Need more help? Reply HELP."}
    return ModelResponse(parts=[ToolCallPart(info.output_tools[0].name, reply)])


with desk.override(model=FunctionModel(promise_then_fix)):
    result = await handle("Refund my cracked mug!", Ticket("Bo"))
for message in result.all_messages():
    for part in message.parts:
        if isinstance(part, RetryPromptPart):
            print("sent back:", part.content)
print("final    :", result.output.message)

assert "guarantee" not in result.output.message.lower()  # R2

> ✅ **Checkpoint** — `sent back: Never promise a refund (you wrote ['guarantee', 'full refund']).
> Say a teammate reviews it within 5 days.`, then `final    : A teammate will review it within 5 days.
> Need more help? Reply HELP.` The promise never reached the customer.

## R3 — a filtered toolset: customers never see the refund tool

The proof habit from 09: `TestModel(call_tools=[])` calls no tools, and
`last_model_request_parameters.function_tools` lists what the model was shown. `{ticket.role:>8}` pads
the role to 8 characters, so the two lines line up.

In [ ]:
def offered(test_model):
    """Names of the tools the model was shown on its last request (from 09)."""
    return [tool.name for tool in test_model.last_model_request_parameters.function_tools]


model = TestModel(call_tools=[])
seen_by = {}
with desk.override(model=model):
    for ticket in [Ticket("Ada"), Ticket("Sam", role="staff")]:
        await handle("Refund order 1042.", ticket)
        seen_by[ticket.role] = offered(model)
        print(f"{ticket.role:>8} sees:", seen_by[ticket.role])

assert "issue_refund" not in seen_by["customer"]  # R3

> ✅ **Checkpoint** — `customer sees: ['lookup_order', 'ask_billing']` and
> `   staff sees: ['lookup_order', 'ask_billing', 'issue_refund']`. Same desk, same message; the deps
> decided the scope.

## R4 — a hook: every tool call is logged

The hook writes to `ctx.deps.audit_log`, so every ticket carries its own log. Ada's run in R1 went through
the hook. The log is sorted because tools asked for in one step run side by side, in no fixed order (10).

In [ ]:
print("Ada's audit log:", sorted(ada.audit_log))

assert sorted(ada.audit_log) == ["ask_billing", "lookup_order"]  # R4

> ✅ **Checkpoint** — `Ada's audit log: ['ask_billing', 'lookup_order']`. Neither tool had to
> remember to log itself: the hook sees every call, including tools you add later.

## R5 — approval: a refund waits for a person

Sam is staff, so R3 shows him `issue_refund`. Plain `TestModel()` calls every tool it is shown, so it
asks for a refund too. The run stops early with a `DeferredToolRequests` (10).

In [ ]:
sam = Ticket("Sam", role="staff")
with desk.override(model=TestModel()):
    paused = await handle("Refund order 1042.", sam)
waiting = [call.tool_name for call in paused.output.approvals]
print("output   :", type(paused.output).__name__)
print("waiting  :", waiting)
print("audit log:", sorted(sam.audit_log))

assert waiting == ["issue_refund"]  # R5

> ✅ **Checkpoint** — `output   : DeferredToolRequests`, `waiting  : ['issue_refund']` and
> `audit log: ['ask_billing', 'lookup_order']`. The refund is not in the log because it has not run.
> Two rows now guard one tool: the filter decides who may *see* it, the approval decides when it may
> *run*. To finish, a person calls `build_results(...)` and you resume the run, as in 10.

## R6 — usage limits: a stuck desk is stopped

This scripted model is stuck: it asks billing over and over. Every loop costs two requests, one for the
desk and one for the billing expert. Both count against the same limit, because `ask_billing` forwards
`usage_limits` (11).

In [ ]:
from pydantic_ai import UsageLimitExceeded


def stuck(messages, info):
    """A scripted model that never stops asking billing."""
    return ModelResponse(parts=[ToolCallPart("ask_billing", {"question": "Refund?"})])


bo = Ticket("Bo")
stopped = None
try:
    with desk.override(model=FunctionModel(stuck)):
        await handle("Refund?", bo)
except UsageLimitExceeded as error:
    stopped = str(error)
print("stopped:", stopped)
print("billing was asked", bo.audit_log.count("ask_billing"), "times")

assert stopped is not None  # R6

> ✅ **Checkpoint** — `stopped: The next request would exceed the request_limit of 6. Consider raising
> the limit…`, then
> `billing was asked 3 times`: 3 desk requests plus 3 billing requests used up the 6. Without the
> limit, the default of 50 would have let it spend 50 (10).

## Steering rules get an eval

The six asserts prove the wiring. R1's rules ("look it up, ask billing, end with HELP") need a real model,
so they get an eval (12). Here is the desk's eval: a `Dataset` whose task runs a ticket through
`handle()`.

- Two built-in checks run on every case: the message is a string, and it contains "Reply HELP".
- `Case(..., evaluators=[...])` adds checks for **that case only**. Each case lists the tools its
  ticket should use, with `ToolCorrectness` (12).

As in 12, test the eval on a fake first: the scripted desk follows the rules by design, so every check
should pass. The exercise adds a case.

In [ ]:
from pydantic_evals import Case, Dataset
from pydantic_evals.evaluators import Contains, IsInstance, ToolCorrectness


async def answer(text: str) -> str:
    """The eval task: one customer ticket in, the desk's message out."""
    result = await handle(text, Ticket("eval"))
    return result.output.message


desk_evals = Dataset(
    name="desk",
    cases=[
        Case(
            name="refund",
            inputs="My mug from order 1042 arrived cracked. I want my money back.",
            evaluators=[ToolCorrectness(expected_tools=["lookup_order", "ask_billing"])],
        ),
        Case(
            name="status",
            inputs="Has order 1042 arrived yet?",
            evaluators=[ToolCorrectness(expected_tools=["lookup_order"])],
        ),
    ],
    evaluators=[IsInstance("str"), Contains("Reply HELP")],
)
with desk.override(model=FunctionModel(fake_desk)):
    report = await desk_evals.evaluate(answer, name="scripted desk", progress=False)
report.print(include_durations=False)

assert report.averages().assertions == 1.0

> ✅ **Checkpoint** — `Evaluation Summary: scripted desk` with two rows. `refund` and `status` each
> show `✔✔✔` (two shared checks plus their own `ToolCorrectness`), and the Averages row says `100.0% ✔`.
> The Metrics column shows `requests: 3` for the refund (billing was asked) and `requests: 2` for the
> status question. To run
> it on a real model, wrap the same `evaluate` in `desk.override(model=MODEL)` and
> `billing_expert.override(model=MODEL)`.

## Live: one real ticket

Same desk, same front door, your `MODEL` in place of both stand-ins. Cleo is a customer with a cracked
mug. Watch each row do its job: the tools she could use (R3), the log (R4), the request count (R6), and a
reply in the `Reply` shape (R2) that ends with the HELP line if the model followed R1.

In [ ]:
from learningdantic import has_key

LIVE = has_key(MODEL)  # True if MODEL's key (or login) is set up. It sends no request.
if LIVE:
    cleo = Ticket("Cleo")
    # Two context managers on one line: both agents use MODEL until the block ends.
    with desk.override(model=MODEL), billing_expert.override(model=MODEL):
        result = await handle(
            "Hi! My mug from order 1042 arrived cracked. Can I get my money back?", cleo
        )
    print(result.output)
    print("audit log:", cleo.audit_log)
    print("requests :", result.usage.requests)
else:
    print("No key found — skipping the live part (the rest of this notebook ran keyless).")

> ✅ **Checkpoint** — with no key: the skip line. With a key: a `Reply` with
> `category='billing'` and a short message about a review within 5 days that ends with
> `Need more help? Reply HELP.`, then (on our run) `audit log: ['lookup_order', 'ask_billing']` and
> `requests : 3`: two for the desk, one for the billing expert. Your wording, and the order in the log,
> will differ. The fences will not: there is no refund in the log, because Cleo can't see that tool.

## 🏋️ Exercise — a new specialist and its eval case

Acme wants late-parcel questions to go to a **shipping expert**. First decide where that rule lives:
"parcel questions go to shipping" is steering (a line in `RULES`, plus the new tool's docstring). The
shipping policy itself lives in the expert's own instructions (11). Add the new tool in the parts cell
and re-run from there, so running a cell twice stays safe.

1. **Add the specialist.** Build a `shipping_expert` agent on `TestModel(custom_output_text=...)` and an
   `ask_shipping` tool in `desk_tools` that runs it and forwards `usage` and `usage_limits`. Assert that a
   customer is shown `ask_shipping` (R3's `offered()` habit), and that a scripted desk model that calls it
   gets the expert's answer into the reply, with `result.usage.requests` counting the expert.
2. **Add its eval case.** Add a `late_parcel` case to `desk_evals` that expects
   `["lookup_order", "ask_shipping"]`. Teach your scripted desk to send parcel tickets to shipping, and
   assert that the report still passes 100%.

*Stretch (optional):* run `desk_evals` on your `MODEL` with the `LIVE` pattern. Does the real model pick
the right expert every time?

*Going further:* add the seventh row of the table, keeping only the last few messages with
`ProcessHistory` (13).

Solution: [`../solutions/14_capstone_extension.ipynb`](../solutions/14_capstone_extension.ipynb)

## Next →

That's the course. You can now build an agent, steer it, scope it and prove both.
Head back to the [README](../README.md) for the full map, the cost notes and where to go next.